## Virtual Environment

Create and activate a virtual environment:

```bash
python -m venv irony
source venv/bin/activate  # On Windows: venv\Scripts\activate
```

or using conda
```
conda create -n irony python=3.14.3
conda activate irony
```
Install kernel

```bash
pip install --user ipykernel
python -m ipykernel install --user --name=irony
```

## Install Dependencies

```bash
pip install -r requirements.txt
```

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32,
    device_map="auto",
    attn_implementation="sdpa"
)

def check_model_loaded(model, tokenizer):
    test_input = "Hello, my name is"
    
    inputs = tokenizer(test_input, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=10)
    
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Input:  {test_input}")
    print(f"Output: {decoded}")
    print(f"Device: {model.device}")
    print(f"Dtype:  {model.dtype}")

check_model_loaded(model, tokenizer)


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.


Input:  Hello, my name is
Output: Hello, my name is Alex, and I'm a software engineer.
Device: mps:0
Dtype:  torch.float32


In [3]:
def run_zero_shot(prompt: str, max_new_tokens: int = 256) -> str:
    # OLMo2 Instruct uses a chat template
    print(prompt)
    messages = [{"role": "user", "content": prompt}]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )
    print(input_ids)
        # If it's a BatchEncoding, unpack it; if it's already a tensor, keep it
    if hasattr(input_ids, "input_ids"):
        input_ids = input_ids["input_ids"]
    
    input_ids = input_ids.to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    response = tokenizer.decode(
        output[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    )
    print(response)
    return response.strip()

In [ ]:
import pandas as pd
from tqdm import tqdm

# Load and process all 3 files
files = {
    "condition1": "./conditions/Condition1(context_richness).csv",
    "condition2": "./conditions/Condition2(common_ground).csv",
    "condition3": "./conditions/Condition3(prompting_style).csv",
}

for name, path in files.items():
    print(f"\nProcessing {name}...")
    
    df = pd.read_csv(path)
    print(df.head())  # sanity check
    
    tqdm.pandas()
    df["model_output"] = df["full_prompt"].progress_apply(run_zero_shot)
    
    output_path = f"results_gemma_zeroshot_{name}.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

print("\nAll done!")


Processing condition1...
      item_id context-level irony_label                 basic_utterance  \
0     C1_01_L           low      ironic  What a great start to the day.   
1  C1_01_L_NI           low  non-ironic  What a great start to the day.   
2     C1_01_H          high      ironic  What a great start to the day.   
3  C1_01_H_NI          high  non-ironic  What a great start to the day.   
4     C1_02_L           low      ironic      That went really smoothly.   

                                         full_prompt  
0  Is the following statement ironic? Answer with...  
1  Is the following statement ironic? Answer with...  
2  Is the following statement ironic? Answer with...  
3  Is the following statement ironic? Answer with...  
4  Is the following statement ironic? Answer with...  


  0%|                                                   | 0/120 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Is the following statement ironic? Answer with yes or no. 
"What a great start to the day."
{'input_ids': tensor([[     2,    105,   2364,    107,   4602,    506,   2269,   5456,  90364,
         236881,  25685,    607,  11262,    653,    951, 236761, 236743,    107,
         236775,   3689,    496,   1822,   1502,    531,    506,   1719,   1781,
            106,    107,    105,   4368,    107]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1]])}


  2%|▋                                          | 2/120 [00:06<06:46,  3.44s/it]

Yes.
Is the following statement ironic? Answer with yes or no. 
"What a great start to the day."
{'input_ids': tensor([[     2,    105,   2364,    107,   4602,    506,   2269,   5456,  90364,
         236881,  25685,    607,  11262,    653,    951, 236761, 236743,    107,
         236775,   3689,    496,   1822,   1502,    531,    506,   1719,   1781,
            106,    107,    105,   4368,    107]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1]])}


...

Yes. 

The statement implies the opposite – that it’s *not* the perfect time – creating an ironic contrast.
Is the following statement ironic? Answer with yes or no. 
"The internet cut out mid-presentation, the slides vanished, and the client had already started leaving. This is the perfect time for this."
{'input_ids': tensor([[     2,    105,   2364,    107,   4602,    506,   2269,   5456,  90364,
         236881,  25685,    607,  11262,    653,    951, 236761, 236743,    107,
         236775,    818,   8379,   3463,    855,   5453, 236772,  46671, 236764,
            506,  23353,  87326, 236764,    532,    506,   4430,   1053,   3016,
           3931,   9542, 236761,   1174,    563,    506,   4011,    990,    573,
            672,   1781,    106,    107,    105,   4368,    107]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1]])}


 63%|██████████████████████████▌               | 76/120 [32:44<19:24, 26.46s/it]

Yes

Is the following statement ironic? Answer with yes or no. 
"Everyone was settled, the room was quiet, and we had a full hour with no interruptions. This is the perfect time for this."
{'input_ids': tensor([[     2,    105,   2364,    107,   4602,    506,   2269,   5456,  90364,
         236881,  25685,    607,  11262,    653,    951, 236761, 236743,    107,
         236775,  40651,    691,  21262, 236764,    506,   2978,    691,  12010,
         236764,    532,    692,   1053,    496,   2587,   6468,    607,    951,
         111383, 236761,   1174,    563,    506,   4011,    990,    573,    672,
           1781,    106,    107,    105,   4368,    107]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]])}


 64%|██████████████████████████▉               | 77/120 [32:49<14:13, 19.85s/it]

Yes

Is the following statement ironic? Answer with yes or no. 
"That driver was so considerate."
{'input_ids': tensor([[     2,    105,   2364,    107,   4602,    506,   2269,   5456,  90364,
         236881,  25685,    607,  11262,    653,    951, 236761, 236743,    107,
         236775,   6372,   7924,    691,    834, 120115,   1781,    106,    107,
            105,   4368,    107]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]])}


In [ ]:
torch.cuda.is_available()
print("Done! Results saved.")